## Silver to Gold Job

Constrói a camada Gold do IMDM (Índice de Maturidade Digital Municipal) — Etapa 3 conforme `docs/arquitetura/gold_dimensional_model.md`. **Os radares não estão no código:** dimensões, itens, regras e pontuação vêm da planilha do especialista (`config/radar/radar_imdm.xlsx`), carregada pelo `pipeline/load_radar_config` nas tabelas `config.radar_*`.

**Vários radares:** um radar (ex.: `A`, `B`) é uma metodologia completa e define todas as edições. Cada execução calcula **todos os radares** da edição do job. As respostas do IBGE não dependem do radar (`fact_resposta`); os pontos, os scores e o IMDM existem uma vez por radar (coluna `radar`).

- 1. Configuração, guarda de pré-requisitos (Ajuste 1, radares carregados) e criação do schema `semantic`
- 2. `dim_municipio` — municípios com porte populacional derivado
- 3. Radares da edição — lidos de `config.radar_*` (dimensões, itens, âncoras e regras de cada radar)
- 4. `dim_indicador` — variáveis com significados e linhagem (glossário; a parametrização dos radares fica em `config.radar_*`)
- 5. `fact_resposta` — despivotização atômica (municípios × indicadores com dados), sem pontuação
- 6. `fact_pontos_item` — pontos e peso de cada item, por radar × município (genérico por regra, pontos de `config.radar_pontuacao`)
- 7. `fact_score_dimensao` — score por radar × município × dimensão
- 8. `fact_imdm_municipio` — IMDM por radar × município (média ponderada das dimensões que entram no IMDM; nível pelas faixas de `config.radar_nivel`)
- 9. Controles automatizados de qualidade (Seção 6), por radar

> **Pré-requisito crítico (Ajuste 1):** a `silver.munic_clean_data` deve preservar os 6 estados textuais originais da MUNIC (`Sim`, `Não`, `-`, `Não sabe informar`, `Não informou`, `Recusa`). O Step 1 bloqueia a execução caso contrário.

> **Pré-requisito dos radares:** o `pipeline/load_radar_config` precisa ter rodado para a edição (linha `radar` no ledger e linhas da edição em `config.radar_*`).


In [0]:
%run ../environment/env_variables

In [0]:
%run ../environment/env_functions

## Step 1 - Configuração, pré-requisitos e schema semântico

- Define as tabelas de origem (silver + diagnósticos `dq_*` da Etapa 2), as tabelas dos radares e as tabelas gold de destino
- Valida que as colunas de indicadores estão em `StringType` com os 6 estados originais preservados (Ajuste 1) — a execução é bloqueada se o cast destrutivo ainda estiver vigente
- Cria o schema `semantic` (Ajuste 3) que receberá as tabelas do notebook `gold_to_semantic`


In [ ]:
from pyspark.sql import functions as F

# Tabelas de origem (silver + diagnósticos da Etapa 2)
silver_data_table  = f"{catalog_name}.{silver_schema}.munic_clean_data"
silver_dict_table  = f"{catalog_name}.{silver_schema}.munic_clean_dict"
dq_profile_table   = f"{catalog_name}.{gold_schema}.dq_column_profile"
dq_skiplogic_table = f"{catalog_name}.{gold_schema}.dq_skiplogic_candidates"

# Radares (definidos na planilha do especialista; carregados pelo load_radar_config)
config_schema          = "config"
radares_table          = f"{catalog_name}.{config_schema}.radares"
radar_dimensao_table   = f"{catalog_name}.{config_schema}.radar_dimensao"
radar_variavel_table   = f"{catalog_name}.{config_schema}.radar_variavel"
radar_pontuacao_table  = f"{catalog_name}.{config_schema}.radar_pontuacao"
radar_nivel_table      = f"{catalog_name}.{config_schema}.radar_nivel"

# Tabelas gold de destino
dim_municipio_table       = f"{catalog_name}.{gold_schema}.dim_municipio"
dim_indicador_table       = f"{catalog_name}.{gold_schema}.dim_indicador"
fact_resposta_table       = f"{catalog_name}.{gold_schema}.fact_resposta"
fact_pontos_item_table    = f"{catalog_name}.{gold_schema}.fact_pontos_item"
fact_score_dimensao_table = f"{catalog_name}.{gold_schema}.fact_score_dimensao"
fact_imdm_table           = f"{catalog_name}.{gold_schema}.fact_imdm_municipio"

# Ajuste 3: schema semântico para as tabelas do gold_to_semantic
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.semantic")

# Carrega a edição corrente da silver
silver_df = spark.table(silver_data_table).filter(F.col("ano_edicao") == reference_year)

# Fronteira silver→gold contra o CONTRATO do ledger (pipeline_manifest — seção 5.3)
manifest_silver = contrato_da_edicao(etapa="silver", ano_edicao=reference_year)
# Radares da edição: o load_radar_config valida a planilha contra esta silver e declara a linha 'radar'
manifest_radar = contrato_da_edicao(etapa="radar", ano_edicao=reference_year)

# Colunas da EDIÇÃO: no schema união da silver, colunas que só existem em outra
# edição chegam 100% nulas para a corrente — a fact_resposta só pode despivotizar
# as colunas com dados (as nulas virariam milhões de células 'não informou' fantasma).
todas_indicadoras = [c for c in silver_df.columns if c.startswith(("mtic", "mgov"))]
indicator_cols = colunas_com_dados(silver_df, todas_indicadoras)   # env_functions (modulos/ledger)

# Check: shape com dados da edição == indicadores declarados pela silver
check_igual(len(indicator_cols), manifest_silver["indicadores"], "silver: indicadores com dados",
              contexto=f"detalhes da silver: {manifest_silver['detalhes']}")
# Check: nº de municípios == linhas declaradas pela silver
check_igual(silver_df.count(), manifest_silver["linhas"], "silver: municípios")
non_string_cols = [c for c in indicator_cols if silver_df.schema[c].dataType.simpleString() != "string"]
if non_string_cols:
    raise Exception(
        "Pré-requisito 'Ajuste 1' NÃO aplicado: colunas de indicadores não são StringType "
        f"(cast destrutivo ainda vigente, ex.: {non_string_cols[:8]}...). "
        "Reexecute o bronze_to_silver preservando os 6 estados originais da MUNIC "
        "('Sim', 'Não', '-', 'Não sabe informar', 'Não informou', 'Recusa') antes de rodar o silver_to_gold."
    )

resumo_validacoes()   # qualquer FAIL interrompe o run aqui (fail-fast), antes de construir a gold


def limpar_edicao(tabela):
    """Apaga as linhas da edição numa tabela por radar antes de regravar.
    O MERGE só atualiza e insere: sem isto, um radar, item ou dimensão que saiu da planilha
    ficaria órfão na gold. Afeta só a edição do job; as outras edições não mudam."""
    if spark.catalog.tableExists(tabela):
        spark.sql(f"DELETE FROM {tabela} WHERE ano_edicao = {reference_year}")


print(f"OK: {len(indicator_cols)} indicadores em StringType (6 estados preservados)")
print(f"Edição {reference_year} | municípios: {silver_df.count()} | planilha dos radares sha256 {manifest_radar['sha256_xlsx'][:12]}…")


## Step 2 - `dim_municipio` (1 linha por município × edição)

- Origem: colunas cadastrais e descritivas da `silver.munic_clean_data`
- Deriva o `porte_municipio` a partir de `populacao_total` (Seção 3.1): Pequeno I (até 20 mil), Pequeno II (20–50 mil), Médio (50–100 mil), Grande I (100–500 mil) e Grande II (mais de 500 mil)

In [0]:
# porte_municipio derivado da população (Seção 3.1 da especificação)
porte_municipio_expr = (
    F.when(F.col("populacao_total") <= 20000, "Pequeno I")
     .when(F.col("populacao_total") <= 50000, "Pequeno II")
     .when(F.col("populacao_total") <= 100000, "Médio")
     .when(F.col("populacao_total") <= 500000, "Grande I")
     .otherwise("Grande II")
)

dim_municipio = silver_df.select(
    "cod_ibge", "ano_edicao", "nome", "uf_sigla", "regiao",
    "populacao_total", "faixa_populacao",
    porte_municipio_expr.alias("porte_municipio"),
)

upsert_delta_table(
    df_source=dim_municipio,
    table_name=dim_municipio_table,
    merge_keys=["cod_ibge", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): dim_municipio com porte populacional derivado",
)
display(dim_municipio.limit(10))

## Step 3 - Radares da edição (lidos de `config.radar_*`)

- Os radares são **definidos pelo especialista** (`config/radar/radar_imdm.xlsx`) e carregados pelo `load_radar_config`, que já os validou contra esta silver — aqui não há lista fixa de radares, itens nem dimensões
- Para cada radar: cada item vale no máximo 1 ponto; cada dimensão = pontos obtidos / pontos possíveis × 100
- Formato interno de cada radar: `(dimensao, variável-âncora do item, regra, variáveis-membro)`; `dimensao` = `dimensao_codigo` (ex.: `servicos_digitais_ao_cidadao`)
- Em itens multi-variável, os pontos ficam na variável-âncora (1ª variável do item na planilha)

**Regras de pontuação:** `SIM_POSITIVO` (Sim = 1), `CATEGORICO_ESCALONADO` (pontos por resposta, ou por variável marcada 'Sim' no formato de publicação), `ESCADA_MAIOR_NIVEL` (maior nível marcado), `GRUPO_PROPORCIONAL` (fração de Sim do grupo)


In [ ]:
# Radares da edição: uma linha por radar × item × variável (config.radar_variavel)
SIM_POS    = "SIM_POSITIVO"
CAT_ESC    = "CATEGORICO_ESCALONADO"
ESCADA     = "ESCADA_MAIOR_NIVEL"
GRUPO      = "GRUPO_PROPORCIONAL"
REGRAS_CONHECIDAS = {SIM_POS, CAT_ESC, ESCADA, GRUPO}
EXCECAO_LAI = "NAO_SE_APLICA_SEM_PEDIDO_LAI"

def _da_edicao(tabela):
    return spark.table(tabela).filter(F.col("edicao") == reference_year)

radares_cad  = spark.table(radares_table).toPandas()
radar_var_ed = _da_edicao(radar_variavel_table).orderBy("radar", "item", "ordem").toPandas()
radar_dim_ed = _da_edicao(radar_dimensao_table).orderBy("radar", "ordem").toPandas()
radar_pts_ed = _da_edicao(radar_pontuacao_table).toPandas()
radar_niv_ed = _da_edicao(radar_nivel_table).orderBy("radar", "minimo").toPandas()

radares_edicao = sorted(radares_cad["radar"])
sem_definicao = [r for r in radares_edicao if r not in set(radar_var_ed["radar"]) or r not in set(radar_dim_ed["radar"])]
if not radares_edicao or sem_definicao:
    raise RuntimeError(
        f"Radares sem definição na edição {reference_year}: {sem_definicao or 'nenhum radar cadastrado'}. "
        "Rode o pipeline/load_radar_config para esta edição (todo radar precisa ter as linhas do ano "
        "em config/radar/radar_imdm.xlsx)."
    )

from collections import Counter

RADARES = {}   # radar -> estrutura usada nos Steps 6 a 9
for rad in radares_edicao:
    rv = radar_var_ed[radar_var_ed["radar"] == rad]
    rd = radar_dim_ed[radar_dim_ed["radar"] == rad]
    grupos = list(rv.groupby("item", sort=False))
    itens = [  # (item, dimensao_codigo, âncora, regra, membros, exceção)
        (item, g["dimensao_codigo"].iloc[0], g["cod_variable"].iloc[0], g["regra"].iloc[0],
         g["cod_variable"].tolist(), g["excecao"].iloc[0] or "")
        for item, g in grupos
    ]
    dims = {
        r.dimensao_codigo: {"nome": r.dimensao, "ordem": int(r.ordem), "peso": float(r.peso),
                            "entra": str(r.entra_no_imdm).lower() == "sim"}
        for r in rd.itertuples()
    }
    # Sanidade (o load_radar_config já validou; aqui garante que gold e radares não divergiram)
    regras_desconhecidas = sorted({i[3] for i in itens} - REGRAS_CONHECIDAS)
    assert not regras_desconhecidas, f"Radar {rad}: regras sem implementação no pipeline: {regras_desconhecidas}"
    dims_sem_cadastro = sorted({i[1] for i in itens} - set(dims))
    assert not dims_sem_cadastro, f"Radar {rad}: itens apontam para dimensões fora de radar_dimensao: {dims_sem_cadastro}"
    membros = [v for i in itens for v in i[4]]
    assert len(membros) == len(set(membros)), f"Radar {rad}: uma variável pertence a mais de um item"
    sem_dados = [v for v in membros if v not in indicator_cols]
    assert not sem_dados, (f"Radar {rad}: variáveis sem dados na silver {reference_year}: {sem_dados} — "
                           "corrija a planilha (o load_radar_config deveria ter barrado).")
    RADARES[rad] = {
        "itens": itens,
        "dims": dims,
        "pontos": radar_pts_ed[radar_pts_ed["radar"] == rad],
        "niveis": radar_niv_ed[radar_niv_ed["radar"] == rad],
        "padrao": bool(radares_cad.loc[radares_cad["radar"] == rad, "padrao"].iloc[0]),
    }
    assert len(RADARES[rad]["niveis"]), f"Radar {rad}: faixas de maturidade da edição {reference_year} ausentes"

check_igual(sum(len(R["itens"]) for R in RADARES.values()), int(manifest_radar["linhas"]),
            f"radares {reference_year}: itens = contrato do load_radar_config")
resumo_validacoes()

for rad, R in RADARES.items():
    contagem = Counter(i[1] for i in R["itens"])
    print(f"Radar {rad}{' (padrão)' if R['padrao'] else ''}: {len(R['itens'])} itens em {len(R['dims'])} dimensões")
    for cod, d in sorted(R["dims"].items(), key=lambda kv: kv[1]["ordem"]):
        print(f"  {d['ordem']}. {d['nome']} ({cod}) — {contagem.get(cod, 0)} itens | peso {d['peso']:g} | entra no IMDM: {d['entra']}")


## Step 4 - `dim_indicador` (1 linha por variável × edição)

- Base do JOIN: as colunas reais da silver via `dq_column_profile` (Ajuste 2) — descarta os códigos fantasmas do dicionário do IBGE e herda os 7 significados reconstruídos na Etapa 2 + a correção de `mtic30`
- `grupo_pergunta`: pai confirmado na `dq_skiplogic_candidates`; caso contrário, radical extraído por regex
- Glossário de negócios das variáveis do IBGE. **A parametrização (que variável entra em que item, dimensão e regra) não fica mais aqui:** ela depende do radar e está em `config.radar_variavel` (uma linha por radar × edição × item × variável)


In [ ]:
# Guard de pré-requisito (Etapa 2 -> Etapa 3): a edição corrente precisa ter sido
# perfilada e PERSISTIDA pelo bronze_quality_profiling (persistir_tabelas='sim').
# As tabelas dq_* acumulam todas as edições (2019, 2024, ...) — a gold só pode usar
# as linhas da edição corrente; sem elas, aborta em vez de usar evidência de outro ano.
def _exigir_dq_da_edicao(nome_tabela):
    if not spark.catalog.tableExists(nome_tabela):
        raise RuntimeError(
            f"Pré-requisito ausente: {nome_tabela} não existe. Rode o notebook "
            f"pipeline/bronze_quality_profiling com persistir_tabelas='sim' "
            f"para a edição {reference_year} antes de construir a gold."
        )
    n = spark.table(nome_tabela).filter(F.col("ano_edicao") == reference_year).count()
    if n == 0:
        raise RuntimeError(
            f"Pré-requisito ausente: {nome_tabela} não tem linhas da edição {reference_year}. "
            f"Rode o notebook pipeline/bronze_quality_profiling com "
            f"persistir_tabelas='sim' para essa edição."
        )
    print(f"Pré-requisito OK: {nome_tabela} — {n} linhas da edição {reference_year}")

_exigir_dq_da_edicao(dq_profile_table)
_exigir_dq_da_edicao(dq_skiplogic_table)

# Base: as colunas reais da silver (via dq_column_profile — Ajuste 2),
# restritas à edição corrente — sem o filtro por ano_edicao, cada variável viria
# de TODAS as edições perfiladas e o MERGE do upsert falharia por chave duplicada
profile_df = (
    spark.table(dq_profile_table)
    .filter(F.col("ano_edicao") == reference_year)
    .filter(F.col("coluna").isin(indicator_cols))
    .select(
        F.col("coluna").alias("cod_variable"),
        F.col("significado"),
        F.col("origem_significado"),
        F.col("tipo_detectado"),
    )
)

# Correção documental do grupo (Ajuste 2, Seção 1.1): o dicionário oficial do IBGE
# traz "educação" em mtic30 por erro de texto; o correto é a área de COMUNICAÇÃO
# (consistente com o bloco mtic31/mtic331 de comunicação social).
CORRECAO_MTIC30 = "Estrutura organizacional para a área de comunicação - existência"
profile_df = profile_df.withColumns(
    {
        "significado": F.when(F.col("cod_variable") == "mtic30", F.lit(CORRECAO_MTIC30)).otherwise(F.col("significado")),
        "origem_significado": F.when(F.col("cod_variable") == "mtic30", F.lit("corrigido pelo grupo")).otherwise(F.col("origem_significado")),
    }
)

# Fallback de significado a partir do dicionário silver
dict_df = (
    spark.table(silver_dict_table)
    .filter(F.col("ano_edicao") == reference_year)
    .select("cod_variable", "meaning")
)

# Pais confirmados por salto lógico (Etapa 2)
# - o bronze_quality_profiling grava o pai em "coluna_pai" (a doc antiga citava "coluna_pai_candidata"):
#   aceita os dois nomes para não quebrar conforme a versão da tabela dq_ no workspace
# - a tabela guarda o melhor pai de CADA filha, inclusive os de relação "parcial": só
#   veredito_dados = 'sim' vira grupo_pergunta; as demais caem no radical por regex
skiplogic_raw = (
    spark.table(dq_skiplogic_table)
    .filter(F.col("ano_edicao") == reference_year)   # só a edição corrente
)
pai_col = next((c for c in ("coluna_pai", "coluna_pai_candidata") if c in skiplogic_raw.columns), None)
assert pai_col, f"{dq_skiplogic_table}: coluna do pai não encontrada (colunas: {skiplogic_raw.columns})"
veredito_col = next((c for c in ("veredito_dados", "relacao_confirmada") if c in skiplogic_raw.columns), None)
if veredito_col:
    skiplogic_raw = skiplogic_raw.filter(F.col(veredito_col) == "sim")
skiplogic_df = skiplogic_raw.select(
    F.col("coluna_filha").alias("cod_variable"),
    F.col(pai_col).alias("pai_confirmado"),
)
print(f"Pais de salto lógico confirmados: {skiplogic_df.count()} (coluna do pai: {pai_col})")

dim_indicador = (
    profile_df
    .join(dict_df, on="cod_variable", how="left")
    .join(skiplogic_df, on="cod_variable", how="left")
    .select(
        "cod_variable",
        F.lit(reference_year).cast("int").alias("ano_edicao"),
        F.coalesce("significado", "meaning").alias("meaning"),
        "origem_significado",
        F.when(F.col("cod_variable").startswith("mtic"), "TIC").otherwise("GOV").alias("tema"),
        F.col("tipo_detectado").alias("tipo_resposta"),
        # grupo_pergunta: pai confirmado na dq_skiplogic_candidates; senão radical por regex
        # (Seção 3.2.1: ex. mtic12a1 -> mtic12). Questão-base sem sufixo não tem pai -> NULL.
        F.coalesce(
            F.when(F.col("pai_confirmado") != "", F.col("pai_confirmado")),
            F.when(
                F.regexp_extract("cod_variable", r"^(mtic|mgov)\d{2}", 0) != F.col("cod_variable"),
                F.regexp_extract("cod_variable", r"^(mtic|mgov)\d{2}", 0),
            ),
        ).alias("grupo_pergunta"),
    )
)

upsert_delta_table(
    df_source=dim_indicador,
    table_name=dim_indicador_table,
    merge_keys=["cod_variable", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): dim_indicador (glossário das variáveis do IBGE)",
)
display(dim_indicador.limit(10))


## Step 5 - `fact_resposta`: despivotização atômica (municípios × indicadores com dados)

- `stack` converte as colunas de indicadores em linhas (município × variável)
- `classe_preenchimento`: as 5 classes mutuamente exclusivas validadas na Etapa 2 — `Valida`, `Nao_Aplicavel_Skip` (salto lógico `-`), `Nao_Sabe`, `Nao_Informou` e `Recusa`
- `resposta_normalizada`: `'1'`/`'0'` para os estados binários; categorias e texto livre preservados; NULL em `Recusa`
- **Não depende de radar:** é o que o IBGE respondeu. Os pontos ficam na `fact_pontos_item` (Step 6), uma vez por radar


In [ ]:
# Despivota as colunas de indicadores em linhas atômicas (município × variável)
stack_pairs = ", ".join(f"'{c}', `{c}`" for c in indicator_cols)
resposta_long = silver_df.select(
    "cod_ibge", "ano_edicao",
    F.expr(f"stack({len(indicator_cols)}, {stack_pairs}) as (cod_variable, resposta_original)"),
).withColumn("resposta_original", F.col("resposta_original").cast("string"))

# "Não sabe" e "Não informou" comparados sem acento, sem caixa e sem espaços nas pontas, para aceitar
# variações de redação entre edições da MUNIC (ex.: "Não sabe informar", "Não sabe", "Não soube informar").
# "Recusa" e "-" continuam com comparação EXATA: a Recusa é o invariante contado desde a bronze
# (celulas_recusa) e conferido no Step 9 contra o contrato da silver.
_SEM_ACENTO = ("ÃãÁáÀàÂâÉéÊêÍíÓóÔôÕõÚúÇç", "AaAaAaAaEeEeIiOoOoOoUuCc")
resposta_chave = F.lower(F.trim(F.translate(F.col("resposta_original"), *_SEM_ACENTO)))
RE_NAO_SABE = r"^nao (sabe|soube)( informar)?$"

# classe_preenchimento: as 5 classes mutuamente exclusivas validadas na Etapa 2
classe_preenchimento_expr = (
    F.when(F.col("resposta_original") == "Recusa", "Recusa")
     .when(resposta_chave.rlike(RE_NAO_SABE), "Nao_Sabe")
     .when(resposta_chave == "nao informou", "Nao_Informou")
     .when(F.col("resposta_original") == "-", "Nao_Aplicavel_Skip")
     .when(F.col("resposta_original").isNull() | (F.trim(F.col("resposta_original")) == ""), "Nao_Informou")
     .otherwise("Valida")
)

# resposta_normalizada: '1'/'0' para estados binários; categoria/texto preservados; NULL em Recusa
# (derivada da classe, para tratar igual qualquer redação dos rótulos de não-resposta)
resposta_normalizada_expr = (
    F.when(F.col("resposta_original") == "Sim", "1")
     .when(F.col("classe_preenchimento") == "Recusa", F.lit(None).cast("string"))
     .when((F.col("resposta_original") == "Não")
           | F.col("classe_preenchimento").isin("Nao_Sabe", "Nao_Informou", "Nao_Aplicavel_Skip"), "0")
     .otherwise(F.col("resposta_original"))
)

fact_resposta = (
    resposta_long
    .withColumn("classe_preenchimento", classe_preenchimento_expr)
    .withColumn("resposta_normalizada", resposta_normalizada_expr)
    .select("cod_ibge", "ano_edicao", "cod_variable",   # ano_edicao em 2º: ordem que o upsert_delta_table mantém
            "resposta_original", "resposta_normalizada", "classe_preenchimento")
)
print(f"Linhas atômicas: {fact_resposta.count()} (esperado: {silver_df.count()} municípios × {len(indicator_cols)} indicadores)")

# Aviso: resposta classificada como 'Valida' mas com cara de não-resposta (rótulo novo numa edição)
suspeitas = (fact_resposta.where((F.col("classe_preenchimento") == "Valida")
                                 & resposta_chave.rlike(r"^nao (sab|soub|inform)|^recus"))
             .groupBy("resposta_original").count().collect())
for r in suspeitas:
    print(f"AVISO: resposta '{r['resposta_original']}' ({r['count']} linhas) ficou como 'Valida' — "
          "se for um rótulo de não-resposta, inclua-o na classificação acima")
if not suspeitas:
    print("OK: nenhum rótulo de não-resposta fora da classificação")

upsert_delta_table(
    df_source=fact_resposta,
    table_name=fact_resposta_table,
    merge_keys=["cod_ibge", "cod_variable", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): fact_resposta atômica (respostas do IBGE, sem pontuação)",
)
display(fact_resposta.limit(10))


## Step 6 - `fact_pontos_item`: pontos de cada item, por radar × município

Genérico por regra — nenhum código de variável escrito aqui; os pontos vêm de `config.radar_pontuacao`. Uma linha por **radar × município × item** (a variável-âncora fica registrada):

- **`SIM_POSITIVO`**: 'Sim' na âncora = 1; demais respostas = 0
- **`CATEGORICO_ESCALONADO` com 1 variável** (ex.: `mtic10`, `mgov09`, `mtic29`, `mgov05`): pontos por resposta exata; resposta fora do mapa pontua 0 e gera AVISO
- **`CATEGORICO_ESCALONADO` com várias variáveis** (formato de publicação) e **`ESCADA_MAIOR_NIVEL`** (canais): vale a maior pontuação entre as variáveis marcadas 'Sim'
- **`GRUPO_PROPORCIONAL`**: fração de 'Sim' entre as variáveis do grupo
- **Exceção `NAO_SE_APLICA_SEM_PEDIDO_LAI`** (item do `mgov05`): 'Não publica' + `mgov04` = 'Não' → sai do denominador (pontos e peso NULL)

Semântica do denominador (Seções 3.3.2 e 3.3.3): `-`, 'Não sabe informar' e 'Não informou' pontuam 0 com peso ativo; `Recusa` na âncora anula pontos e peso.


In [ ]:
import re
import unicodedata
from functools import reduce

ESTADOS_NAO_PONTUAVEIS = {"Recusa", "Não sabe informar", "Não informou", "-"}

def _eh_nao_sabe(valor):
    return re.match(RE_NAO_SABE, unicodedata.normalize("NFKD", valor).encode("ascii", "ignore").decode().strip().lower())

# Valores distintos das âncoras escalonadas de 1 variável (para avisar rótulo sem pontuação)
ancoras_rotulo = sorted({i[2] for R in RADARES.values() for i in R["itens"] if i[3] == CAT_ESC and len(i[4]) == 1})
valores_rotulo = {}
if ancoras_rotulo:
    linha = silver_df.agg(*[F.collect_set(c).alias(c) for c in ancoras_rotulo]).collect()[0].asDict()
    valores_rotulo = {c: set(v) for c, v in linha.items()}

avisos_step6 = []
partes = []
for rad, R in RADARES.items():
    tabela_pts = R["pontos"]
    structs = []
    for item, dim, anchor, regra, members, excecao in R["itens"]:
        is_recusa = F.col(anchor) == "Recusa"
        nao_se_aplica = F.lit(False)
        if excecao == EXCECAO_LAI:
            if "mgov04" in indicator_cols:
                nao_se_aplica = (F.col(anchor) == "Não publica") & (F.col("mgov04") == "Não")
            else:
                avisos_step6.append(f"Radar {rad}: mgov04 sem dados na edição {reference_year} — exceção do 'não publica' ({anchor}) não avaliável")

        if regra == SIM_POS:
            pontos = F.when(F.col(anchor) == "Sim", F.lit(1.0)).otherwise(F.lit(0.0))
        elif regra == GRUPO:
            sim_count = reduce(lambda a, b: a + b,
                               [F.when(F.col(c) == "Sim", F.lit(1.0)).otherwise(F.lit(0.0)) for c in members])
            pontos = sim_count / F.lit(float(len(members)))
        else:
            g = tabela_pts[tabela_pts["item"] == item]
            mapa = {(r.cod_variable, r.resposta): float(r.pontos) for r in g.itertuples()}
            if len(members) == 1:
                # Escalonado de 1 variável: pontos por resposta exata
                nao_cobertos = sorted(v for v in valores_rotulo.get(anchor, set()) - {r for (_, r) in mapa}
                                      - ESTADOS_NAO_PONTUAVEIS if not _eh_nao_sabe(v))
                if nao_cobertos:
                    avisos_step6.append(f"Radar {rad}, {anchor}: valores fora do mapa do radar — pontuam 0.0 até mapeamento: {nao_cobertos}")
                pontos = F.coalesce(*[F.when(F.col(anchor) == r, F.lit(p)) for (_, r), p in mapa.items()], F.lit(0.0))
            else:
                # Várias variáveis (formato de publicação / escada): maior pontuação entre as marcadas 'Sim'
                por_var = [F.when(F.col(v) == "Sim", F.lit(p)).otherwise(F.lit(0.0)) for (v, _), p in mapa.items()]
                pontos = F.greatest(*por_var) if len(por_var) > 1 else por_var[0]

        fora = is_recusa | nao_se_aplica
        structs.append(F.struct(
            F.lit(item).alias("item"),
            F.lit(dim).alias("dimensao_codigo"),
            F.lit(anchor).alias("variavel_ancora"),
            F.lit(regra).alias("regra"),
            F.when(fora, F.lit(None).cast("double")).otherwise(pontos.cast("double")).alias("pontos_obtidos"),
            F.when(fora, F.lit(None).cast("double")).otherwise(F.lit(1.0)).alias("peso_aplicado"),
        ))
    # Um select por radar: os itens viram linhas (explode de um array de structs)
    partes.append(
        silver_df.select("cod_ibge", "ano_edicao", F.lit(rad).alias("radar"), F.explode(F.array(*structs)).alias("p"))
        .select("radar", "cod_ibge", "ano_edicao", "p.*")
    )

for a in avisos_step6:
    print(f"AVISO: {a}")

fact_pontos_item = reduce(lambda a, b: a.unionByName(b), partes)

limpar_edicao(fact_pontos_item_table)
upsert_delta_table(
    df_source=fact_pontos_item,
    table_name=fact_pontos_item_table,
    merge_keys=["radar", "cod_ibge", "ano_edicao", "item"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): pontos e peso de cada item, por radar × município",
)
fact_pontos_item = spark.table(fact_pontos_item_table).filter(F.col("ano_edicao") == reference_year)
for rad, R in RADARES.items():
    n = fact_pontos_item.where(F.col("radar") == rad).select("item").distinct().count()
    print(f"Radar {rad}: {n} itens pontuados (esperado pelo radar: {len(R['itens'])})")
display(fact_pontos_item.limit(10))


## Step 7 - `fact_score_dimensao` (1 linha por radar × município × dimensão)

- `Score_d = (Σ pontos_obtidos / Σ peso_aplicado) × 100` sobre os itens da dimensão no radar
- Tabela longa: o nº de dimensões varia por radar e por edição
- Toda combinação município × dimensão do radar existe (score NULL quando não há item no denominador)


In [ ]:
dims_pdf = spark.createDataFrame(
    [(rad, cod, d["nome"], d["ordem"], d["peso"], d["entra"])
     for rad, R in RADARES.items() for cod, d in R["dims"].items()],
    "radar string, dimensao_codigo string, dimensao_nome string, dimensao_ordem int, dimensao_peso double, entra_no_imdm boolean",
)

somas = (
    fact_pontos_item
    .groupBy("radar", "cod_ibge", "ano_edicao", "dimensao_codigo")
    .agg(F.sum("pontos_obtidos").alias("pontos_obtidos"), F.sum("peso_aplicado").alias("pontos_possiveis"))
)
fact_score_dimensao = (
    silver_df.select("cod_ibge", "ano_edicao").crossJoin(dims_pdf)
    .join(somas, on=["radar", "cod_ibge", "ano_edicao", "dimensao_codigo"], how="left")
    .withColumn(
        "score",
        F.when(F.col("pontos_possiveis") > 0,
               F.round(F.col("pontos_obtidos") / F.col("pontos_possiveis") * 100, 2))
         .otherwise(F.lit(None).cast("double")),
    )
    .select("radar", "cod_ibge", "ano_edicao", "dimensao_codigo", "dimensao_nome", "dimensao_ordem",
            "dimensao_peso", "entra_no_imdm", "pontos_obtidos", "pontos_possiveis", "score")
)

limpar_edicao(fact_score_dimensao_table)
upsert_delta_table(
    df_source=fact_score_dimensao,
    table_name=fact_score_dimensao_table,
    merge_keys=["radar", "cod_ibge", "ano_edicao", "dimensao_codigo"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): score por radar × município × dimensão",
)
fact_score_dimensao = spark.table(fact_score_dimensao_table).filter(F.col("ano_edicao") == reference_year)
display(fact_score_dimensao.limit(10))


## Step 8 - `fact_imdm_municipio` (1 linha por radar × município)

- `IMDM = Σ (peso_d × Score_d) / Σ peso_d` sobre as dimensões do radar com `entra_no_imdm = sim` (pesos da planilha; pesos iguais = média simples)
- NULL: `Recusa` (município com recusa formal em alguma variável da edição — vale para todos os radares) ou `Sem_Item_Valido` (alguma dimensão do IMDM sem item válido naquele radar)
- `nivel_maturidade_mmd`: faixas do radar na edição, em `config.radar_nivel` (rótulo `"<nível>-<nome>"`)


In [ ]:
# Recusa formal: qualquer variável da edição com 'Recusa' (não depende do radar)
flag_recusa = (
    fact_resposta.groupBy("cod_ibge", "ano_edicao")
    .agg(F.max(F.when(F.col("classe_preenchimento") == "Recusa", 1).otherwise(0)).alias("flg_recusa"))
)

radares_df = spark.createDataFrame(
    [(rad, sum(1 for d in R["dims"].values() if d["entra"])) for rad, R in RADARES.items()],
    "radar string, n_dimensoes_imdm int",
)
imdm_agg = (
    fact_score_dimensao.where(F.col("entra_no_imdm"))
    .groupBy("radar", "cod_ibge", "ano_edicao")
    .agg(
        F.sum(F.col("dimensao_peso") * F.col("score")).alias("soma_ponderada"),
        F.sum(F.when(F.col("score").isNotNull(), F.col("dimensao_peso"))).alias("soma_pesos"),
        F.count("score").alias("n_dimensoes_pontuadas"),
    )
)

# Nível de maturidade: faixas de cada radar na edição (o último nível de cada radar inclui o máximo, 100)
nivel_expr = F.lit(None).cast("string")
for rad, R in RADARES.items():
    niveis = R["niveis"].sort_values("minimo").to_dict("records")
    for i, n in enumerate(reversed(niveis)):
        rotulo = f"{int(n['nivel'])}-{n['nome']}"
        cond = (F.col("radar") == rad) & (F.col("score_global_imdm") >= float(n["minimo"])) & (
            (F.col("score_global_imdm") <= float(n["maximo"])) if i == 0 else (F.col("score_global_imdm") < float(n["maximo"]))
        )
        nivel_expr = F.when(cond, F.lit(rotulo)).otherwise(nivel_expr)

fact_imdm_municipio = (
    flag_recusa.crossJoin(radares_df)
    .join(imdm_agg, on=["radar", "cod_ibge", "ano_edicao"], how="left")
    .withColumn(
        "score_global_imdm",
        F.when((F.col("flg_recusa") == 0) & (F.col("n_dimensoes_pontuadas") == F.col("n_dimensoes_imdm")),
               F.round(F.col("soma_ponderada") / F.col("soma_pesos"), 2))
         .otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "motivo_imdm_nulo",
        F.when(F.col("flg_recusa") == 1, F.lit("Recusa"))
         .when(F.col("score_global_imdm").isNull(), F.lit("Sem_Item_Valido"))
         .otherwise(F.lit(None).cast("string")),
    )
    .withColumn("nivel_maturidade_mmd", F.when(F.col("score_global_imdm").isNotNull(), nivel_expr))
    .withColumn("radar_sha256", F.lit(manifest_radar["sha256_xlsx"]))
    .select(
        "radar", "cod_ibge", "ano_edicao", "motivo_imdm_nulo", "score_global_imdm", "nivel_maturidade_mmd",
        "n_dimensoes_imdm", F.coalesce("n_dimensoes_pontuadas", F.lit(0)).alias("n_dimensoes_pontuadas"),
        "radar_sha256",
    )
)

limpar_edicao(fact_imdm_table)
upsert_delta_table(
    df_source=fact_imdm_municipio,
    table_name=fact_imdm_table,
    merge_keys=["radar", "cod_ibge", "ano_edicao"],
    ano_edicao=reference_year,
    description="Etapa 3 (silver_to_gold): IMDM por radar (média ponderada das dimensões do radar) e estágio MMD",
)
display(fact_imdm_municipio.limit(10))


## Step 9 - Controles automatizados de qualidade (Seção 6)

Todos os esperados vêm da edição corrente e de **cada radar** (nenhum nº de radares, itens ou dimensões escrito no código):

1. **Respostas (iguais para todos os radares):** nº de variáveis e temas na `dim_indicador`; `municípios × indicadores` linhas na `fact_resposta`, Recusa preservada contra o contrato da silver e classes fechando o total
2. **Pontos, por radar:** `municípios × itens` na `fact_pontos_item`, itens por regra, peso máximo por dimensão = nº de itens da dimensão
3. **IMDM, por radar:** `fact_score_dimensao` = municípios × dimensões; IMDM NULL = recusas + municípios sem item válido em alguma dimensão do índice; recálculo independente pela média ponderada; scores em [0, 100]
4. **Integridade referencial:** 0 órfãos nas FKs fato → dimensões; toda âncora existe na `fact_resposta`
5. **Proteção ética:** `mgov15`/`mgov16` fora de todos os radares

Qualquer FAIL interrompe o pipeline com `AssertionError`.


In [ ]:
dim_municipio_gold  = spark.table(dim_municipio_table).filter(F.col("ano_edicao") == reference_year)
dim_indicador_gold = spark.table(dim_indicador_table).filter(F.col("ano_edicao") == reference_year)
fact_resposta_gold = spark.table(fact_resposta_table).filter(F.col("ano_edicao") == reference_year)
fact_pontos_gold   = spark.table(fact_pontos_item_table).filter(F.col("ano_edicao") == reference_year)
fact_score_gold    = spark.table(fact_score_dimensao_table).filter(F.col("ano_edicao") == reference_year)
fact_imdm_gold     = spark.table(fact_imdm_table).filter(F.col("ano_edicao") == reference_year)

checks = []
def check(controle, esperado, obtido):
    checks.append((controle, str(esperado), str(obtido), "PASS" if esperado == obtido else "FAIL"))

munic_count     = manifest_silver["linhas"]
recusa_contrato = manifest_silver["recusa_total"]
recusa_munis = fact_resposta_gold.where(F.col("classe_preenchimento") == "Recusa").select("cod_ibge").distinct()
n_recusa_munis = recusa_munis.count()

# ---- 1) Respostas (não dependem do radar)
tema_counts = {r["tema"]: r["count"] for r in dim_indicador_gold.groupBy("tema").count().collect()}
temas_esperado = {
    "TIC": sum(1 for c in indicator_cols if c.startswith("mtic")),
    "GOV": sum(1 for c in indicator_cols if c.startswith("mgov")),
}
check(f"dim_indicador: {len(indicator_cols)} variáveis (colunas com dados da edição)", len(indicator_cols), dim_indicador_gold.count())
check(f"dim_indicador: temas TIC/GOV {temas_esperado}", temas_esperado, tema_counts)
reconstruidas = ["mtic11", "mtic31", "mtic331", "mtic2421", "mtic2431", "mgov141", "mgov142"]
reconstruidas_edicao = [c for c in reconstruidas if c in indicator_cols]
check(
    f"Ajuste 2: significados preenchidos nas {len(reconstruidas_edicao)} reconstruídas presentes na edição",
    0,
    dim_indicador_gold.where(F.col("cod_variable").isin(reconstruidas_edicao) & F.col("meaning").isNull()).count(),
)
if "mtic30" in indicator_cols:
    mtic30_meaning = dim_indicador_gold.where(F.col("cod_variable") == "mtic30").select("meaning").collect()[0]["meaning"]
    check("Ajuste 2: mtic30 corrigido para 'comunicação'", True,
          (mtic30_meaning is not None) and ("comunicação" in mtic30_meaning.lower()))
else:
    print(f"mtic30 sem dados na edição {reference_year} — correção documental não aplicável (skip)")
check("Ajuste 3.2.1: grupo_pergunta sem strings vazias", 0, dim_indicador_gold.where(F.col("grupo_pergunta") == "").count())

classe_dist = {r["classe_preenchimento"]: r["count"]
               for r in fact_resposta_gold.groupBy("classe_preenchimento").count().collect()}
esperado_linhas = munic_count * len(indicator_cols)
check(f"fact_resposta: {esperado_linhas} linhas ({munic_count} × {len(indicator_cols)})", esperado_linhas, fact_resposta_gold.count())
check("fact_resposta: Recusa preservada (contrato da silver/edição)", recusa_contrato, classe_dist.get("Recusa", 0))
check("fact_resposta: classes fecham o total (5 classes mutuamente exclusivas)", esperado_linhas, sum(classe_dist.values()))
check("FK: fact_resposta -> dim_municipio (0 órfãos)", 0,
      fact_resposta_gold.join(dim_municipio_gold.select("cod_ibge", "ano_edicao"), ["cod_ibge", "ano_edicao"], "left_anti").count())
check("FK: fact_resposta -> dim_indicador (0 órfãos)", 0,
      fact_resposta_gold.join(dim_indicador_gold.select("cod_variable", "ano_edicao"), ["cod_variable", "ano_edicao"], "left_anti").count())

# Proteção ética/privacidade de servidores: mgov15/mgov16 em nenhum radar
eticas_edicao = [c for c in ("mgov15", "mgov16") if c in indicator_cols]
check(f"Ética: mgov15/mgov16 fora de todos os radares ({len(eticas_edicao)} presentes na edição)", 0,
      int(radar_var_ed["cod_variable"].isin(eticas_edicao).sum()))

check(f"fact_pontos_item/score/imdm: radares gravados = radares cadastrados {radares_edicao}",
      radares_edicao,
      sorted(r[0] for r in fact_imdm_gold.select("radar").distinct().collect()))

# ---- 2) e 3) Por radar
for rad, R in RADARES.items():
    pre = f"[{rad}]"
    itens, dims = R["itens"], R["dims"]
    p = fact_pontos_gold.where(F.col("radar") == rad)
    s = fact_score_gold.where(F.col("radar") == rad)
    m = fact_imdm_gold.where(F.col("radar") == rad)

    check(f"{pre} fact_pontos_item: {munic_count} × {len(itens)} itens", munic_count * len(itens), p.count())
    esperado_regras = dict(Counter(i[3] for i in itens))
    regra_counts = {r["regra"]: r["count"] for r in p.select("item", "regra").distinct().groupBy("regra").count().collect()}
    check(f"{pre} itens por regra {esperado_regras}", esperado_regras, regra_counts)
    esperado_peso_dim = {d: float(n) for d, n in Counter(i[1] for i in itens).items()}
    peso_max_dim = {r["dimensao_codigo"]: r["peso_max"] for r in (
        p.groupBy("cod_ibge", "dimensao_codigo").agg(F.sum("peso_aplicado").alias("peso"))
         .groupBy("dimensao_codigo").agg(F.max("peso").alias("peso_max")).collect())}
    check(f"{pre} peso máximo por dimensão = nº de itens {esperado_peso_dim}", esperado_peso_dim, peso_max_dim)
    check(f"{pre} pontos entre 0 e peso (cada item vale no máximo 1)", 0,
          p.where((F.col("pontos_obtidos") < 0) | (F.col("pontos_obtidos") > F.col("peso_aplicado"))).count())
    check(f"{pre} fora do denominador (peso NULL) sem pontos", 0,
          p.where(F.col("peso_aplicado").isNull() & F.col("pontos_obtidos").isNotNull()).count())
    check(f"{pre} FK: âncoras existem na fact_resposta (0 órfãos)", 0,
          p.select("cod_ibge", "ano_edicao", F.col("variavel_ancora").alias("cod_variable"))
           .join(fact_resposta_gold.select("cod_ibge", "ano_edicao", "cod_variable"),
                 ["cod_ibge", "ano_edicao", "cod_variable"], "left_anti").count())

    dims_imdm = [c for c, d in dims.items() if d["entra"]]
    check(f"{pre} fact_score_dimensao: {munic_count} × {len(dims)} dimensões", munic_count * len(dims), s.count())
    sem_item_valido = (
        s.where(F.col("entra_no_imdm") & F.col("score").isNull()).select("cod_ibge").distinct()
        .join(recusa_munis, "cod_ibge", "left_anti").count()
    )
    nulos = m.where(F.col("score_global_imdm").isNull())
    check(f"{pre} fact_imdm: {munic_count} municípios", munic_count, m.count())
    check(f"{pre} fact_imdm: IMDM NULL = {n_recusa_munis} com Recusa + {sem_item_valido} sem item válido",
          n_recusa_munis + sem_item_valido, nulos.count())
    check(f"{pre} fact_imdm: motivo 'Recusa' nos municípios com recusa", n_recusa_munis,
          nulos.where(F.col("motivo_imdm_nulo") == "Recusa").count())
    stats = m.where(F.col("score_global_imdm").isNotNull()).agg(
        F.count("*").alias("n"), F.min("score_global_imdm").alias("mn"), F.max("score_global_imdm").alias("mx")).collect()[0]
    check(f"{pre} fact_imdm: municípios válidos = total - NULL", munic_count - n_recusa_munis - sem_item_valido, stats["n"])
    check(f"{pre} fact_imdm: scores dentro de [0, 100]", True,
          stats["n"] == 0 or ((stats["mn"] >= 0) and (stats["mx"] <= 100)))
    check(f"{pre} fact_imdm: todo IMDM válido tem nível de maturidade", 0,
          m.where(F.col("score_global_imdm").isNotNull() & F.col("nivel_maturidade_mmd").isNull()).count())
    recalc = (
        s.where(F.col("entra_no_imdm"))
        .groupBy("cod_ibge")
        .agg((F.sum(F.col("dimensao_peso") * F.col("score")) / F.sum("dimensao_peso")).alias("imdm_recalc"),
             F.count("score").alias("n"))
        .where(F.col("n") == len(dims_imdm))
        .join(m.select("cod_ibge", "score_global_imdm"), "cod_ibge")
        .where(F.col("score_global_imdm").isNotNull())
        .where(F.abs(F.col("imdm_recalc") - F.col("score_global_imdm")) > 0.011)
        .count()
    )
    check(f"{pre} fact_imdm: IMDM = média ponderada das dimensões (recálculo independente)", 0, recalc)
    for nome_fato, df_fato in (("fact_pontos_item", p), ("fact_score_dimensao", s), ("fact_imdm", m)):
        check(f"{pre} FK: {nome_fato} -> dim_municipio (0 órfãos)", 0,
              df_fato.join(dim_municipio_gold.select("cod_ibge", "ano_edicao"), ["cod_ibge", "ano_edicao"], "left_anti").count())

results = spark.createDataFrame(checks, schema="controle string, esperado string, obtido string, status string")
display(results)

failed = [c[0] for c in checks if c[3] == "FAIL"]
assert not failed, f"Controles de qualidade da camada Gold REPROVADOS: {failed}"
print(f"Todos os {len(checks)} controles de qualidade da Etapa 3 (Seção 6) passaram ({len(RADARES)} radar(es)).")


In [ ]:
# Declara o contrato da GOLD no ledger e fecha o invariante da cadeia: a recusa
# que nasceu na bronze, preservada pela silver, chega intacta à fact_resposta
# (docs/arquitetura/pipeline_manifest.md, seções 4 e 8).
n_pass = sum(1 for c in checks if c[3] == "PASS")

gravar_linha_manifest(
    etapa="gold",
    ano_edicao=reference_year,
    tabela_destino=";".join([dim_municipio_table, dim_indicador_table, fact_resposta_table,
                             fact_pontos_item_table, fact_score_dimensao_table, fact_imdm_table]),
    linhas=int(munic_count),                         # municípios da edição (as fatos por radar = municípios × radares)
    indicadores=int(dim_indicador_gold.count()),
    recusa_total=int(classe_dist.get("Recusa", 0)),
    detalhes={
        "checks_pass": f"{n_pass}/{len(checks)}",
        "fact_resposta_linhas": str(fact_resposta_gold.count()),
        "radares": ", ".join(RADARES),
        "radar_padrao": ", ".join(r for r, R in RADARES.items() if R["padrao"]),
        "itens_por_radar": " | ".join(f"{r}: {len(R['itens'])}" for r, R in RADARES.items()),
        "dimensoes_por_radar": " || ".join(
            f"{r}: " + " | ".join(f"{d['ordem']}. {d['nome']}{'' if d['entra'] else ' (fora do IMDM)'}"
                                 for d in sorted(R["dims"].values(), key=lambda d: d["ordem"]))
            for r, R in RADARES.items()),
        "imdm_nulos_por_radar": " | ".join(
            f"{r}: {fact_imdm_gold.where((F.col('radar') == r) & F.col('score_global_imdm').isNull()).count()}"
            for r in RADARES),
        "radar_sha256": manifest_radar["sha256_xlsx"],
    },
)
